# Session 1 · Part 2 — QC, filtering, and normalization

**Goal:** make every data-processing decision visible. We calculate RNA and protein QC separately,
visualize the distributions, apply transparent data-adaptive filters, normalize RNA with library-size
scaling + `log1p`, and normalize protein with a centered log-ratio (CLR) transform.

The quantile threshold below is a compact teaching default, not a universal biological cutoff. Inspect
the figures and adapt it to the assay, tissue, and expected cell/spot content.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter inside the hands-on_tutorial directory.")

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


## 1. Calculate spot-level QC metrics


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.plotting import plot_qc_overview, plot_raw_vs_normalized
from dgat_tutorial.processing import (
    calculate_qc_metrics, choose_qc_thresholds, filter_modalities,
    normalize_total_log1p, clr_normalize, validate_modalities,
)

dataset = load_tutorial_data(paths.raw_data)
spots = dataset.spots.copy()
transcripts = dataset.transcripts.select_dtypes(include=[np.number]).copy()
proteins = dataset.proteins.select_dtypes(include=[np.number]).copy()
validate_modalities(spots, transcripts, proteins)
qc_before = calculate_qc_metrics(transcripts, proteins)
qc_before.describe().T


### Figure 1 — RNA and protein QC distributions


In [ ]:
qc_figure, _ = plot_qc_overview(qc_before)
qc_figure_path = paths.figures / "session01_qc_rna_and_protein.png"
qc_figure.savefig(qc_figure_path, dpi=160, bbox_inches="tight")
plt.show()


**How to read it:** low RNA total and low detected-gene counts flag weak transcript capture; protein
totals and detected proteins reveal a different assay channel and therefore need separate inspection.
Large upper tails can be biological or technical—do not remove them automatically without context.


## 2. Choose and apply explicit filters


In [ ]:
LOWER_QUANTILE = 0.01
thresholds = choose_qc_thresholds(qc_before, lower_quantile=LOWER_QUANTILE)
filtered_spots, filtered_rna, filtered_protein, keep_mask = filter_modalities(
    spots, transcripts, proteins, thresholds,
    min_spots_per_gene=1, min_spots_per_protein=1,
)
filtering_summary = pd.DataFrame([{
    **thresholds,
    "spots_before": len(spots), "spots_after": len(filtered_spots),
    "genes_before": transcripts.shape[1], "genes_after": filtered_rna.shape[1],
    "proteins_before": proteins.shape[1], "proteins_after": filtered_protein.shape[1],
}])
filtering_summary.T


## 3. Normalize each modality for its measurement process


In [ ]:
rna_normalized = normalize_total_log1p(filtered_rna, target_sum=10_000)
protein_normalized = clr_normalize(filtered_protein)

normalization_checks = pd.DataFrame({
    "quantity": ["RNA target depth before log1p", "mean CLR protein value per spot"],
    "expected": [10_000.0, 0.0],
    "observed_median": [
        float(np.expm1(rna_normalized).sum(axis=1).median()),
        float(protein_normalized.mean(axis=1).median()),
    ],
})
normalization_checks


### Figure 2 — What normalization changes


In [ ]:
rna_feature = filtered_rna.var(axis=0).idxmax()
protein_feature = filtered_protein.var(axis=0).idxmax()
rna_fig, _ = plot_raw_vs_normalized(filtered_rna, rna_normalized, rna_feature, "RNA")
protein_fig, _ = plot_raw_vs_normalized(filtered_protein, protein_normalized, protein_feature, "protein")
rna_norm_path = paths.figures / "session01_rna_raw_vs_normalized.png"
protein_norm_path = paths.figures / "session01_protein_raw_vs_normalized.png"
rna_fig.savefig(rna_norm_path, dpi=160, bbox_inches="tight")
protein_fig.savefig(protein_norm_path, dpi=160, bbox_inches="tight")
plt.show()


## 4. Save processed matrices and a QC audit trail


In [ ]:
qc_path = paths.results / "session01_spot_qc.csv"
filtering_path = paths.results / "session01_filtering_summary.csv"
rna_path = paths.processed_data / "rna_log_normalized.csv"
protein_path = paths.processed_data / "protein_clr_normalized.csv"
qc_before.assign(kept=keep_mask).to_csv(qc_path)
filtering_summary.to_csv(filtering_path, index=False)
rna_normalized.to_csv(rna_path)
protein_normalized.to_csv(protein_path)
manifest = write_checkpoint(
    "1.2", [qc_path, filtering_path, rna_path, protein_path, qc_figure_path, rna_norm_path, protein_norm_path],
    summary={"spots_kept": len(filtered_spots), "spots_removed": int((~keep_mask).sum())}, start=paths.root,
)
print(f"Checkpoint written: {manifest}")


## Check

You should now be able to justify: (1) which spots/features were removed, (2) why RNA and protein
use different transforms, and (3) why raw values must remain available for QC and auditability.
